# GitHub Copilot Workshop · Microsoft Fabric

GitHub Copilot is the coding assistant; Fabric is the execution target. Synthetic manufacturing data only. See `docs/notebooks.md` before running. Use a disposable sandbox: data notebooks perform guarded full refreshes, not production ingestion.


In [ ]:
workspace_id = ""
lakehouse_id = ""
sandbox = False
approval = ""
batch_id = ""


In [ ]:
"""Fail-closed Fabric notebook authorization, also embedded by the builder."""

import re
from uuid import UUID


def require_uuid(value: str, label: str) -> str:
    if not isinstance(value, str):
        raise ValueError(f"{label} must be an explicit canonical UUID")
    try:
        parsed = UUID(value)
    except (ValueError, AttributeError) as error:
        raise ValueError(f"{label} must be an explicit canonical UUID") from error
    if str(parsed) != value or parsed.int == 0:
        raise ValueError(f"{label} must be a nonzero lowercase canonical UUID")
    return value


def assert_sandbox_context(runtime_context, *, workspace_id: str, lakehouse_id: str,
                           sandbox: bool, approval: str, batch_id: str) -> None:
    """Names, implicit current-workspace defaults, and truthy strings are not consent."""
    require_uuid(workspace_id, "workspace_id")
    require_uuid(lakehouse_id, "lakehouse_id")
    if sandbox is not True or approval != workspace_id:
        raise ValueError("Require sandbox=True and approval equal to the exact workspace UUID")
    if not isinstance(batch_id, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", batch_id):
        raise ValueError("batch_id must be a nonempty, path-safe run identifier (at most 80 characters)")
    try:
        current_workspace = runtime_context["currentWorkspaceId"]
        default_lakehouse = runtime_context["defaultLakehouseId"]
    except (KeyError, TypeError, AttributeError) as error:
        raise ValueError("Fabric runtime workspace and default lakehouse UUIDs are required") from error
    if current_workspace != workspace_id or default_lakehouse != lakehouse_id:
        raise ValueError("Fabric runtime UUIDs do not match the explicitly approved sandbox")
    attached_workspace = runtime_context.get("defaultLakehouseWorkspaceId")
    if attached_workspace and attached_workspace != workspace_id:
        raise ValueError("Cross-workspace default lakehouses are not supported by this workshop")


def assert_model_target(*, workspace_id: str, model_workspace_id: str,
                        expected_model_id: str) -> None:
    require_uuid(model_workspace_id, "model_workspace_id")
    require_uuid(expected_model_id, "expected_model_id")
    if model_workspace_id != workspace_id:
        raise ValueError("The model must be in the explicitly approved sandbox workspace")

def require_sandbox():
    assert_sandbox_context(
        notebookutils.runtime.context, workspace_id=workspace_id, lakehouse_id=lakehouse_id,
        sandbox=sandbox, approval=approval, batch_id=batch_id,
    )

require_sandbox()


In [ ]:
"""Embedded Spark helpers for this disposable, synthetic full-refresh path only."""

from collections.abc import Callable
from datetime import UTC, datetime

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

RUN_STARTED = datetime.now(UTC)
RUN_OUTPUTS = {}
SYNTHETIC_TABLES = {
    "02_bronze_ingest": {
        f"bronze.{name}" for name in (
            "dim_asset", "dim_customer", "dim_date", "dim_defect_type", "dim_line", "dim_plant",
            "dim_product", "dim_shift", "fact_anomaly_event", "fact_customer_order",
            "fact_maintenance_event", "fact_production_run", "fact_quality_inspection",
        )
    },
    "03_silver_transform": {
        f"silver.{name}" for name in (
            "fact_production_run", "fact_quality_inspection", "fact_maintenance_event",
            "fact_customer_order", "dim_date", "dim_shift", "dim_line", "dim_asset",
            "dim_defect_type", "dq_rejects",
        )
    },
    "04_silver_dims_catchup": {"silver.dim_product", "silver.dim_customer", "silver.dim_plant"},
    "05_gold_star_schema": {
        f"gold.{name}" for name in (
            "dim_date", "dim_plant", "dim_line", "dim_asset", "dim_shift", "dim_product",
            "dim_customer", "dim_defect_type", "fact_production_run", "fact_quality_inspection",
            "fact_maintenance_event", "fact_customer_order", "agg_plant_month",
        )
    },
}


def configure_lineage(spark_session: SparkSession, guard: Callable[[], None], run_batch_id: str,
                      notebook_name: str, source_revision: str):
    """Bind the explicit notebook context after its parameter and UUID guard cells."""
    global spark, require_sandbox, batch_id, NOTEBOOK_NAME, SOURCE_REVISION
    guard()
    if notebook_name not in SYNTHETIC_TABLES:
        raise ValueError("Unknown synthetic notebook")
    spark, require_sandbox, batch_id = spark_session, guard, run_batch_id
    NOTEBOOK_NAME, SOURCE_REVISION = notebook_name, source_revision


def add_bronze_lineage(df):
    return (df.withColumn("_source_file", F.col("_metadata.file_path"))
            .withColumn("_ingest_timestamp", F.current_timestamp())
            .withColumn("_source_system", F.lit("synthetic-manufacturing-csv"))
            .withColumn("_batch_id", F.lit(batch_id)))


def add_silver_lineage(df):
    business_columns = sorted(column for column in df.columns if not column.startswith("_"))
    return (df.withColumn("_processed_timestamp", F.current_timestamp())
            .withColumn("_record_hash", F.sha2(F.to_json(F.struct(*[
                F.col(column) for column in business_columns
            ]), options={"ignoreNullFields": "false"}), 256))
            .withColumn("_batch_id", F.lit(batch_id)))


def write_synthetic(df, table):
    require_sandbox()
    if table not in SYNTHETIC_TABLES[NOTEBOOK_NAME]:
        raise ValueError("Output is outside this notebook's fixed synthetic table allowlist")
    if table.startswith("gold."):
        df = df.withColumn("_built_timestamp", F.current_timestamp()).withColumn("_batch_id", F.lit(batch_id))
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(table)
    rows = spark.table(table).count()
    RUN_OUTPUTS[table] = rows
    return rows


def emit_run_metadata():
    require_sandbox()
    if set(RUN_OUTPUTS) != SYNTHETIC_TABLES[NOTEBOOK_NAME]:
        raise ValueError("Not all expected outputs were written; do not mark this notebook successful")
    spark.sql("CREATE SCHEMA IF NOT EXISTS workshop")
    rows_written = sum(RUN_OUTPUTS.values())
    rows_quarantined = RUN_OUTPUTS.get("silver.dq_rejects", 0)
    if NOTEBOOK_NAME.startswith("02_"):
        rows_read = rows_written
    else:
        source_layer = "silver" if NOTEBOOK_NAME.startswith("05_") else "bronze"
        inputs = [table.split(".", 1)[1] for table in RUN_OUTPUTS
                  if not table.endswith(".dq_rejects") and not table.endswith(".agg_plant_month")]
        rows_read = sum(spark.table(f"{source_layer}.{table}").count() for table in inputs)
    metadata = [(
        batch_id, NOTEBOOK_NAME, NOTEBOOK_NAME.split("_")[1], "synthetic_manufacturing",
        RUN_STARTED, datetime.now(UTC), rows_read, rows_written, rows_quarantined,
        "succeeded", 1, SOURCE_REVISION,
    )]
    audit = spark.createDataFrame(metadata, """
        batch_id string, notebook string, layer string, domain string,
        started_at timestamp, ended_at timestamp, rows_read long, rows_written long, rows_quarantined long,
        status string, dictionary_version int, source_git_sha string
    """)
    audit.write.mode("append").format("delta").saveAsTable("workshop._pipeline_runs")
    print(f"Recorded successful run {batch_id}; stop the Spark session when finished.")

configure_lineage(spark, require_sandbox, batch_id, '04_silver_dims_catchup', 'd0d8c57a5dfc3d9eff8a2339c0266395367733df')


# Lab 4 · Catch-up: build the three Silver dimensions in code

Run after 03 as the reproducible code route for these three dimensions. A separately reviewed Dataflow
Gen2 may produce the same business columns, but do not execute both writers concurrently:

| Table | Rows | Cleaning applied (same steps as the Dataflow) |
|---|---|---|
| `silver.dim_product` | 20 | trim names; `Capitalize Each Word` category; strip `$` and `,` from price; remove the duplicate key |
| `silver.dim_customer` | 35 | trim text; `SG/MY/IN/AU` → full country name; `Capitalize Each Word` country; remove the duplicate key |
| `silver.dim_plant` | 4 | trim text; upper-case the plant code |

The generated notebook injects the shared UUID guard and processing lineage.


In [ ]:
require_sandbox()
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")


def trim_all(df):
    return df.select(*[F.trim(F.col(c)).alias(c) for c in df.columns if not c.startswith("_")])


def save(df, table):
    return write_synthetic(add_silver_lineage(df), table)


## dim_product

The order matters: **clean first, then remove duplicates.** The duplicate row for one product only matches its
twin once the text is trimmed.


In [ ]:
require_sandbox()
product = (
    trim_all(spark.table("bronze.dim_product"))
    .withColumn("Category", F.initcap("Category"))
    .withColumn("UnitPrice", F.regexp_replace("UnitPrice", "[$,]", ""))
    .select(
        F.col("ProductKey").cast("bigint").alias("ProductKey"),
        "ProductCode", "ProductName", "ProductFamily", "Category",
        F.col("UnitPrice").cast("double").alias("UnitPrice"),
        F.col("StandardCost").cast("double").alias("StandardCost"),
    )
    .dropDuplicates(["ProductKey"])
)
print("silver.dim_product:", save(product, "silver.dim_product"), "rows")


## dim_customer


In [ ]:
require_sandbox()
country_codes = {"SG": "Singapore", "MY": "Malaysia", "IN": "India", "AU": "Australia"}
country_fix = F.col("Country")
for code, name in country_codes.items():
    country_fix = F.when(F.col("Country") == code, name).otherwise(country_fix)

customer = (
    trim_all(spark.table("bronze.dim_customer"))
    .withColumn("Country", F.initcap(country_fix))
    .select(
        F.col("CustomerKey").cast("bigint").alias("CustomerKey"),
        "CustomerName", "Industry", "Segment", "Region", "Country",
    )
    .dropDuplicates(["CustomerKey"])
)
print("silver.dim_customer:", save(customer, "silver.dim_customer"), "rows")


## dim_plant


In [ ]:
require_sandbox()
plant = (
    trim_all(spark.table("bronze.dim_plant"))
    .withColumn("PlantCode", F.upper("PlantCode"))
    .select(
        F.col("PlantKey").cast("bigint").alias("PlantKey"),
        "PlantCode", "PlantName", "Country", "Region", "PlantType", "PlantManager",
    )
)
print("silver.dim_plant:", save(plant, "silver.dim_plant"), "rows")


## Verify


In [ ]:
require_sandbox()
checks = {
    "dim_product": (20, "ProductKey"),
    "dim_customer": (35, "CustomerKey"),
    "dim_plant": (4, "PlantKey"),
}
for table, (expected, key) in checks.items():
    df = spark.table(f"silver.{table}")
    rows, keys = df.count(), df.select(key).distinct().count()
    print(f"{'✅' if rows == expected == keys else '❌'} silver.{table:14s} {rows} rows, {keys} distinct keys (expected {expected})")
    if rows != expected or keys != expected:
        raise ValueError(f"silver.{table}: expected {expected} unique keys and rows, got {keys}/{rows}")

countries = sorted(r["Country"] for r in spark.table("silver.dim_customer").select("Country").distinct().collect())
plant_codes = sorted(r["PlantCode"] for r in spark.table("silver.dim_plant").collect())
print("countries:", countries)
print("plant codes:", plant_codes)
assert countries == ["Australia", "India", "Malaysia", "Singapore"], countries
assert plant_codes == ["AU-01", "IN-01", "MY-01", "SG-01"], plant_codes
print("\n🎉 Silver dimensions ready. On to Lab 5 (Gold).")


In [ ]:
require_sandbox()
emit_run_metadata()
